In [18]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

#### In previous notebook, we saved a copy of the universe dataset in folder "data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"

It has every row of every symbol plus an `in_universe` column. The data we research on is the rows where `in_universe` is True.

In [19]:
PROJECT_ROOT = Path.cwd().parent.parent
data_path = PROJECT_ROOT/"data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"
df = pd.read_parquet(data_path)

#### Check length of data

In [20]:
# the period of the universe: only the rows flagged in the universe count
members = df[df["in_universe"]]
start_date = members["close_time"].min()
end_date = members["close_time"].max()

print(f"Start date: {str(start_date)}")
print(f"End data: {str(end_date)}")
print(f"Amount of data: {str(end_date-start_date)}")
print(f"Amounf of data in years: {1299/365}")

Start date: 2023-03-28 23:59:59.999000+00:00
End data: 2026-10-17 23:59:59.999000+00:00
Amount of data: 1299 days 00:00:00
Amounf of data in years: 3.558904109589041


#### We have 1299 days of data

We will want to split the data into two main sections:
- Train set
- Test set (holdout)

I want the holdout period to be around 1 year (365 days)\
The holdout period will be seperated from the rest of the data and will not be opened until the final unbiased evaluation.

## Regular walk forward cross validation

In [21]:
# Proposed split of all data

# Train % of data, Holdout % of data = 70%, 30%

#
# Train    : ==============       (909.3 days)
# Holdout  :               ====== (389.7 days)
# Total    : ==================== (1299 days)



Within the Train set we will also split the data further into more folds

In [22]:
# Proposed split of the train set data

# if train data index = 0, 20
# fold 1 train : [:8]
# fold 1 eval  : [8:14]
# fold 2 train : [:14]
# fold 2 eval  : [14:]


# for the folds:
# = denotes data used for training
# + denotes data used for evaluation

# Fold 1    : ========++++++       (8 units of data used for training, 6 used for evaluation) (train: 363.72 days) (eval: 272.79 days)
# Fold 2    : ==============++++++ (14 units of data used for training, 6 used for evaluation) (train: 636.51 days) (eval: 272.79 days)
# Total     : ==================== (20 units total) (934 days)

# Conclusion

- We will perform all model research and development on the train-set, the holdout set is only for a final unbiased evaluation
- The splits are cut by **date**, not by row position. The file is sorted by symbol first, so slicing it by rows would cut by symbol instead of by time
- The labels look up to 7 days ahead, so a training row whose label ends after its evaluation period starts overlaps the evaluation period. When the labels are added (next notebook) we purge them: keep the training rows with `fwd_end_time_7 < start of the evaluation period`

In [23]:
#  We have decided on (cut by date, as proportions of the days that have symbols in the universe):

# every day with at least one symbol in the universe, oldest first
days = df.loc[df["in_universe"], "open_time"].drop_duplicates().sort_values().reset_index(drop=True)

# train = first 70% of the days, holdout = last 30% of the days
holdout_start = days.iloc[int(len(days) * 0.70)]
train_days = days[days < holdout_start]

#  fold 1 train = open_time <  fold1_eval_start
#  fold 1 eval  = fold1_eval_start <= open_time < fold2_eval_start
#  fold 2 train = open_time <  fold2_eval_start
#  fold 2 eval  = fold2_eval_start <= open_time < holdout_start
fold1_eval_start = train_days.iloc[int(len(train_days) * 0.4)]
fold2_eval_start = train_days.iloc[int(len(train_days) * 0.7)]

print(f"Fold 1 eval starts : {fold1_eval_start}")
print(f"Fold 2 eval starts : {fold2_eval_start}")
print(f"Holdout starts     : {holdout_start}")

Fold 1 eval starts : 2024-03-25 00:00:00+00:00
Fold 2 eval starts : 2024-12-23 00:00:00+00:00
Holdout starts     : 2025-09-22 00:00:00+00:00
